In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(f'{path}/Q1_data.csv')

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df.drop('Order_ID', axis=1, inplace=True)
df.head()

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
categoricalC = df.select_dtypes(include=['object']).columns
numericalC = df.select_dtypes(include=['float', 'int']).columns

for col in categoricalC:
  df[col] = df[col].fillna(df[col].mode()[0])

for col in numericalC:
  df[col] = df[col].fillna(df[col].mean())

print('After handling missing values (categorical by mode, numerical by mean)')
check_missing_values(df)

df.info()

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)
df.info()

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder

for col in categoricalC:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])

df.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

features = df.columns.drop("Delivery_Time")
print(features)

scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])

In [ ]:
df.head()

In [ ]:
# Task 6: Write your code here:




# Since we train for regression we won't have classes to look for their balance

In [ ]:
# Task 1: Write your code here: Split the dataset into features (X) and target (y)

X = df[features]
y = df['Delivery_Time']

print('Input shape: ', X.shape)
print('Target shape: ', y.shape)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error


model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

kf= KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

for fold, (trainIdx, testIdx) in enumerate(kf.split(X)):
  xtrain, xtest = X.iloc[trainIdx], X.iloc[testIdx]
  ytrain, ytest = y.iloc[trainIdx], y.iloc[testIdx]


  model.fit(xtrain, ytrain)
  ypred = model.predict(xtest)


  mae_score = mean_absolute_error(ytest, ypred)
  mae_scores.append(mae_score)
  print(f'Fold {fold+1}/5 |   MAE score: {mae_score:.4f}')


print(f'\nAverage MAE score across all folds is: {np.mean(mae_scores):.4f} ')


In [ ]:
# Task 1: Write your code here:
importance = model.feature_importances_

features = X.columns
sorted_idx = np.argsort(importance)


plt.barh(features[sorted_idx], importance[sorted_idx])
plt.title("Random Forest Regressor Feature Importance")
plt.xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

ypred = pd.DataFrame({
    "Delivery_Time": ypred
})

ypred.hist()
plt.grid(False)
plt.show()

In [ ]:
%pip install catboost

In [ ]:
# Task Bonus: Write your code here:
from catboost import CatBoostRegressor

models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1),
  "CatBoost": CatBoostRegressor(verbose=0)
  }

kf= KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

for fold, (trainIdx, testIdx) in enumerate(kf.split(X)):
  xtrain, xtest = X.iloc[trainIdx], X.iloc[testIdx]
  ytrain, ytest = y.iloc[trainIdx], y.iloc[testIdx]

  ypreds = []
  for model_name, model in models.items():
    print(f"Training {model_name}...")
    model.fit(xtrain, ytrain)
    ypred = model.predict(xtest)
    ypreds.append(ypred)


  ypreds = np.mean(ypreds)

  mae_score = mean_absolute_error(ytest, ypred)
  mae_scores.append(mae_score)
  print(f'\nFold {fold+1}/5 |   MAE score (avg both model): {mae_score:.4f}\n')

print('\n'+ '-'*30)
print(f'Average MAE score across all folds is: {np.mean(mae_scores):.4f} ')
print('-'*30)



In [ ]:
print('Average MAE between the model is better than having one model')